<a href="https://colab.research.google.com/github/lekhasree15/AI_LAB/blob/main/lab3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import heapq

class PuzzleState:
    def __init__(self, board, parent=None, move="", g=0):
        self.board = board
        self.parent = parent
        self.move = move
        self.g = g
        self.h = self.calculate_heuristic()
        self.f = self.g + self.h

    def calculate_heuristic(self):
        goal = (1, 2, 3, 4, 5, 6, 7, 8, 0)
        misplaced = 0
        for i in range(9):
            if self.board[i] != 0 and self.board[i] != goal[i]:
                misplaced += 1
        return misplaced

    def get_neighbors(self):
        neighbors = []
        zero_idx = self.board.index(0)
        row, col = zero_idx // 3, zero_idx % 3

        moves = [(-1, 0, "Up"), (1, 0, "Down"), (0, -1, "Left"), (0, 1, "Right")]

        for dr, dc, move_name in moves:
            nr, nc = row + dr, col + dc
            if 0 <= nr < 3 and 0 <= nc < 3:
                neighbor_idx = nr * 3 + nc
                new_board = list(self.board)
                new_board[zero_idx], new_board[neighbor_idx] = new_board[neighbor_idx], new_board[zero_idx]
                neighbors.append(PuzzleState(tuple(new_board), self, move_name, self.g + 1))
        return neighbors

    def __lt__(self, other):
        return self.f < other.f

def solve_a_star_misplaced(start_board):
    start_state = PuzzleState(start_board)
    goal_board = (1, 2, 3, 4, 5, 6, 7, 8, 0)

    open_set = []
    heapq.heappush(open_set, start_state)
    closed_set = set()

    print("--- A* Search Process (Misplaced Tiles) ---")
    step = 0

    while open_set:
        current = heapq.heappop(open_set)
        step += 1
        print(f"\nIteration {step}:")
        print(f"Exploring State: {current.board[:3]}, {current.board[3:6]}, {current.board[6:]} (g={current.g}, h={current.h}, f={current.f})")

        if current.board == goal_board:
            path = []
            curr = current
            while curr.parent:
                path.append((curr.move, curr.board))
                curr = curr.parent
            return path[::-1], current.g

        closed_set.add(current.board)
        neighbors = current.get_neighbors()
        print(f"  Generated neighbors/paths:")

        for neighbor in neighbors:
            status = "Already visited" if neighbor.board in closed_set else "Added to open set"
            print(f"    -> Move {neighbor.move} to state {neighbor.board[:3]}, {neighbor.board[3:6]}, {neighbor.board[6:]} | g={neighbor.g}, h={neighbor.h}, f={neighbor.f} ({status})")
            if neighbor.board in closed_set:
                continue
            heapq.heappush(open_set, neighbor)

    return None, -1

start_layout = (1, 2, 3, 0, 4, 6, 7, 5, 8)
path, cost = solve_a_star_misplaced(start_layout)
print(f"\nSolution Found in {cost} moves!")
for step, (move, board) in enumerate(path, 1):
    print(f"Step {step}: Move {move} -> {board[:3]}, {board[3:6]}, {board[6:]}")

--- A* Search Process (Misplaced Tiles) ---

Iteration 1:
Exploring State: (1, 2, 3), (0, 4, 6), (7, 5, 8) (g=0, h=3, f=3)
  Generated neighbors/paths:
    -> Move Up to state (0, 2, 3), (1, 4, 6), (7, 5, 8) | g=1, h=4, f=5 (Added to open set)
    -> Move Down to state (1, 2, 3), (7, 4, 6), (0, 5, 8) | g=1, h=4, f=5 (Added to open set)
    -> Move Right to state (1, 2, 3), (4, 0, 6), (7, 5, 8) | g=1, h=2, f=3 (Added to open set)

Iteration 2:
Exploring State: (1, 2, 3), (4, 0, 6), (7, 5, 8) (g=1, h=2, f=3)
  Generated neighbors/paths:
    -> Move Up to state (1, 0, 3), (4, 2, 6), (7, 5, 8) | g=2, h=3, f=5 (Added to open set)
    -> Move Down to state (1, 2, 3), (4, 5, 6), (7, 0, 8) | g=2, h=1, f=3 (Added to open set)
    -> Move Left to state (1, 2, 3), (0, 4, 6), (7, 5, 8) | g=2, h=3, f=5 (Already visited)
    -> Move Right to state (1, 2, 3), (4, 6, 0), (7, 5, 8) | g=2, h=3, f=5 (Added to open set)

Iteration 3:
Exploring State: (1, 2, 3), (4, 5, 6), (7, 0, 8) (g=2, h=1, f=3)
  Gener

In [5]:
import heapq

class PuzzleStateManhattan:
    def __init__(self, board, parent=None, move="", g=0):
        self.board = board
        self.parent = parent
        self.move = move
        self.g = g
        self.h = self.calculate_heuristic()
        self.f = self.g + self.h

    def calculate_heuristic(self):
        goal_positions = {
            1: (0, 0), 2: (0, 1), 3: (0, 2),
            4: (1, 0), 5: (1, 1), 6: (1, 2),
            7: (2, 0), 8: (2, 1)
        }
        manhattan = 0
        for i, val in enumerate(self.board):
            if val != 0:
                current_row, current_col = i // 3, i % 3
                goal_row, goal_col = goal_positions[val]
                manhattan += abs(current_row - goal_row) + abs(current_col - goal_col)
        return manhattan

    def get_neighbors(self):
        neighbors = []
        zero_idx = self.board.index(0)
        row, col = zero_idx // 3, zero_idx % 3
        moves = [(-1, 0, "Up"), (1, 0, "Down"), (0, -1, "Left"), (0, 1, "Right")]

        for dr, dc, move_name in moves:
            nr, nc = row + dr, col + dc
            if 0 <= nr < 3 and 0 <= nc < 3:
                neighbor_idx = nr * 3 + nc
                new_board = list(self.board)
                new_board[zero_idx], new_board[neighbor_idx] = new_board[neighbor_idx], new_board[zero_idx]
                neighbors.append(PuzzleStateManhattan(tuple(new_board), self, move_name, self.g + 1))
        return neighbors

    def __lt__(self, other):
        return self.f < other.f

def solve_a_star_manhattan(start_board):
    start_state = PuzzleStateManhattan(start_board)
    goal_board = (1, 2, 3, 4, 5, 6, 7, 8, 0)

    open_set = []
    heapq.heappush(open_set, start_state)
    closed_set = set()

    print("--- A* Search Process (Manhattan Distance) ---")
    step = 0

    while open_set:
        current = heapq.heappop(open_set)
        step += 1
        print(f"\nIteration {step}:")
        print(f"Exploring State: {current.board[:3]}, {current.board[3:6]}, {current.board[6:]} (g={current.g}, h={current.h}, f={current.f})")

        if current.board == goal_board:
            path = []
            curr = current
            while curr.parent:
                path.append((curr.move, curr.board))
                curr = curr.parent
            return path[::-1], current.g

        closed_set.add(current.board)
        neighbors = current.get_neighbors()
        print(f"  Generated neighbors/paths:")

        for neighbor in neighbors:
            status = "Already visited" if neighbor.board in closed_set else "Added to open set"
            print(f"    -> Move {neighbor.move} to state {neighbor.board[:3]}, {neighbor.board[3:6]}, {neighbor.board[6:]} | g={neighbor.g}, h={neighbor.h}, f={neighbor.f} ({status})")
            if neighbor.board in closed_set:
                continue
            heapq.heappush(open_set, neighbor)

    return None, -1

start_layout = (1, 2, 3, 0, 4, 6, 7, 5, 8)
path, cost = solve_a_star_manhattan(start_layout)
print(f"\nSolution Found in {cost} moves!")
for step, (move, board) in enumerate(path, 1):
    print(f"Step {step}: Move {move} -> {board[:3]}, {board[3:6]}, {board[6:]}")

--- A* Search Process (Manhattan Distance) ---

Iteration 1:
Exploring State: (1, 2, 3), (0, 4, 6), (7, 5, 8) (g=0, h=3, f=3)
  Generated neighbors/paths:
    -> Move Up to state (0, 2, 3), (1, 4, 6), (7, 5, 8) | g=1, h=4, f=5 (Added to open set)
    -> Move Down to state (1, 2, 3), (7, 4, 6), (0, 5, 8) | g=1, h=4, f=5 (Added to open set)
    -> Move Right to state (1, 2, 3), (4, 0, 6), (7, 5, 8) | g=1, h=2, f=3 (Added to open set)

Iteration 2:
Exploring State: (1, 2, 3), (4, 0, 6), (7, 5, 8) (g=1, h=2, f=3)
  Generated neighbors/paths:
    -> Move Up to state (1, 0, 3), (4, 2, 6), (7, 5, 8) | g=2, h=3, f=5 (Added to open set)
    -> Move Down to state (1, 2, 3), (4, 5, 6), (7, 0, 8) | g=2, h=1, f=3 (Added to open set)
    -> Move Left to state (1, 2, 3), (0, 4, 6), (7, 5, 8) | g=2, h=3, f=5 (Already visited)
    -> Move Right to state (1, 2, 3), (4, 6, 0), (7, 5, 8) | g=2, h=3, f=5 (Added to open set)

Iteration 3:
Exploring State: (1, 2, 3), (4, 5, 6), (7, 0, 8) (g=2, h=1, f=3)
  Ge